# benchmarking_main.ipynb
**Scalable Benchmarking Framework for Cell Type Annotation Tools (Python Version)**

This is the main execution notebook that orchestrates the benchmarking process:
1. Loads configuration and tool registry
2. Sources helper functions from benchmarking_helpers.ipynb
3. Provides main execution function run_all_tools_cv()
4. Contains example usage patterns

Supports both marker-based and reference-based methods using k-fold cross-validation with comprehensive result aggregation.

This is a Python/scanpy translation of the R Benchmarking.R file.

## Load Required Libraries

In [ ]:
# Load required libraries
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import time
import warnings
from typing import Dict, List, Tuple, Any, Optional, Callable
import os
import pickle
from datetime import datetime

## Load Helper Functions

Note: In a Jupyter environment, you would either:
1. Run the benchmarking_helpers.ipynb notebook first, or
2. Import the functions if converted to a .py module

For now, we'll assume the helper functions are available.

In [ ]:
# Load helper functions
%run benchmarking_helpers.ipynb  # Uncomment if running in Jupyter

# Alternative: import from .py version
# from benchmarking_helpers import (
#     calculate_metrics, create_cv_folds, get_fold_data_cached,
#     run_single_tool_cv, aggregate_tool_results, aggregate_all_tools
# )

# Import DL-based tool functions — each wrapped individually so one
# missing/broken file does not silence all other tools.
import sys
sys.path.append('./DL-based')

def _try_import(module_name, func_name):
    try:
        mod = __import__(module_name, fromlist=[func_name])
        fn = getattr(mod, func_name)
        print(f'  ✓ {module_name}')
        return fn
    except Exception as e:
        print(f'  ⚠ {module_name}: {e}')
        return None

print('Importing DL-based tools:')
run_scDeepSort_function    = _try_import('run_scDeepSort',    'run_scDeepSort_function')
run_scHash_function        = _try_import('run_scHash',        'run_scHash_function')
run_scDeepinsight_function = _try_import('run_scDeepinsight', 'run_scDeepinsight_function')
run_TOSICA_function        = _try_import('run_TOSICA',        'run_TOSICA_function')
run_scMMT_function         = _try_import('run_scMMT',         'run_scMMT_function')
run_scnym_function         = _try_import('run_scnym',         'run_scnym_function')
run_CIForm_function        = _try_import('run_CIForm',        'run_CIForm_function')
run_mtANN_function         = _try_import('run_mtANN',         'run_mtANN_function')
run_scTransSort_function   = _try_import('run_scTransSort',   'run_scTransSort_function')
run_scBalance_function     = _try_import('run_scBalance',     'run_scBalance_function')
run_MARS_function          = _try_import('run_MARS',          'run_MARS_function')
run_Cell_BLAST_function    = _try_import('run_Cell_BLAST',    'run_Cell_BLAST_function')
run_scArches_function      = _try_import('run_scArches',      'run_scArches_function')
run_SCANVI_function        = _try_import('run_SCANVI',        'run_SCANVI_function')
run_ItClust_function       = _try_import('run_ItClust',       'run_ItClust_function')
run_ACTINN_function        = _try_import('run_ACTINN',        'run_ACTINN_function')
run_scGAD_function         = _try_import('run_scGAD',         'run_scGAD_function')
run_scSemiCluster_function = _try_import('run_scSemiCluster', 'run_scSemiCluster_function')
run_scSemiGAN_function     = _try_import('run_scSemiGAN',     'run_scSemiGAN_function')
run_C2S_function           = _try_import('run_C2S', 'run_C2S_function')

#Not working: scDeepInsight (too complex), scTransSort(broken), mtANN (broken/too long), ItClust(too old)
# scnym (incompatible, too slow)

## Tool Registry and Execution

In [ ]:
# Tool registry - populate with converted tool functions
TOOL_REGISTRY = {}

if run_scDeepSort_function is not None:
    TOOL_REGISTRY["scDeepSort"] = run_scDeepSort_function

if run_scHash_function is not None:
    TOOL_REGISTRY["scHash"] = run_scHash_function

if run_scDeepinsight_function is not None:
    TOOL_REGISTRY["scDeepinsight"] = run_scDeepinsight_function

if run_TOSICA_function is not None:
    TOOL_REGISTRY["TOSICA"] = run_TOSICA_function

if run_scMMT_function is not None:
    TOOL_REGISTRY["scMMT"] = run_scMMT_function

if run_scnym_function is not None:
    TOOL_REGISTRY["scnym"] = run_scnym_function

if run_CIForm_function is not None:
    TOOL_REGISTRY["CIForm"] = run_CIForm_function

if run_mtANN_function is not None:
    TOOL_REGISTRY["mtANN"] = run_mtANN_function

if run_scTransSort_function is not None:
    TOOL_REGISTRY["scTransSort"] = run_scTransSort_function

if run_scBalance_function is not None:
    TOOL_REGISTRY["scBalance"] = run_scBalance_function

if run_MARS_function is not None:
    TOOL_REGISTRY["MARS"] = run_MARS_function

if run_Cell_BLAST_function is not None:
    TOOL_REGISTRY["Cell_BLAST"] = run_Cell_BLAST_function

if run_scArches_function is not None:
    TOOL_REGISTRY["scArches"] = run_scArches_function

if run_SCANVI_function is not None:
    TOOL_REGISTRY["SCANVI"] = run_SCANVI_function    

if run_ItClust_function is not None:
    TOOL_REGISTRY["ItClust"] = run_ItClust_function

if run_ACTINN_function is not None:
    TOOL_REGISTRY["ACTINN"] = run_ACTINN_function

if run_scGAD_function is not None:
    TOOL_REGISTRY["scGAD"] = run_scGAD_function

if run_scSemiCluster_function is not None:
    TOOL_REGISTRY["scSemiCluster"] = run_scSemiCluster_function
    
if run_scSemiGAN_function is not None:
    TOOL_REGISTRY["scSemiGAN"] = run_scSemiGAN_function
    
if run_C2S_function is not None:
    TOOL_REGISTRY["C2S"] = run_C2S_function

print(f"Tools registered: {list(TOOL_REGISTRY.keys())}")
print(f"Total DL tools available: {len(TOOL_REGISTRY)}")

# Example Usage

## MODE 1: K-Fold Cross-Validation

In [ ]:
# 1. Load data and config 
# adata = sc.read_h5ad(ADATA_PATH)

#CV_CONFIG = {
#    'k': 3,                              # Number of folds
#    'group_by': None,                    # For grouped CV, set to None for stratified
#    'stratify_by': "Ground_Truth_Celltype", # For stratified CV
#    'seed': 123                          # Reproducibility
#}

# 2. Run benchmarking
# results = run_all_tools_cv(adata, tools_to_run=TOOLS_TO_RUN)

# 3. Aggregate and compare
# final_results = aggregate_all_tools(results)

# 4. View results
# print(final_results['comparison_table'])

# 5. Save results
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# final_results['comparison_table'].to_csv(f"results_cv_{timestamp}.csv", index=False)


## MODE 2: Single 80/20 Split

In [ ]:
# 1. Load data and config
adata = sc.read_h5ad("./data/Darmanis-Brain-2015_for_use_subset.h5ad")

SPLIT_CONFIG = {
    'label_col':   'Ground_Truth_Celltype',
    'train_prop':  0.80,
    'seed':        123
}

TOOLS_TO_RUN = ["C2S"]

# 2. Run benchmarking
results = run_all_tools_single_split(adata, tools_to_run=TOOLS_TO_RUN)

# 3. Aggregate and compare
final_results = aggregate_all_tools(results)

# 4. View results
print(final_results['comparison_table'])

# 5. Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
final_results['comparison_table'].to_csv(f"results1_cv_{timestamp}.csv", index=False)

## MODE 5: Separate Datasets

In [ ]:
# 1. Load training and testing data
# adata_train = sc.read_h5ad('path/to/train.h5ad')
# adata_test = sc.read_h5ad('path/to/test.h5ad')

# 2. Run benchmarking (cross-dataset generalization)
# results = run_separate_datasets(adata_train, adata_test, tools_to_run=TOOLS_TO_RUN)

# 3. Aggregate and compare
# final_results = aggregate_all_tools(results)

# 4. View results
# print(final_results['comparison_table'])